# Datamine POLREG Process Example

This notebook demonstrates how to configure and run the **`polreg`** process wrapper in `dmstudio`.

## Process Description

## POLREG

Polynomial regression. The polynomial fitted is up to the 5th order, and thus includes a regression line.

The polynomial is of the form:-

## Y = C0 + C1*X + C2*X2 + C3*X3 + C4*X4 + C5*X5

where the coefficients C0, C1, C2, C3, C4 and C5 are written to the output file.

The input file must contain two explicit numeric fields *X and *Y. The polynomial is calculated as Y estimates for X values. There must be at least (order+1) points. If either the *X or *Y value is absent data in a particular record, the record is ignored. This allows regression polynomials to be computed on incomplete sets of data; for example original assays and a partial set of check assays.

* **DATAFILE** : Name of the input data file.

* **XDATA** : The X field in the input file that was used.

* **YDATA** : The Y field in the input data file that was used.

* **GOODNFIT** : The goodness of fit.

* **CORRCOEF** : The coefficient of variance.

* **STDERR** : The standard error.

### Input Files:

* **in** (Undefined):
  Input file.
  Required=Yes

### Output Files:

* **out** (Undefined):
  Output file containing the coefficients.
  Required=No

### Fields:

* **x** (Numeric : IN):
  X co-ordinate of the sample data.
  Default=X
  Required=Yes

* **y** (Numeric : IN):
  Y co-ordinate of the sample data.
  Default=Y
  Required=Yes

### Parameters:

* **order**:
  Order of the polynomial required (1,2,3,4 or 5).
  Range=1,5
  Values=1,2,3,4,5
  Default=1
  Required=Yes

* **print**:
  If set to 1 then a table of estimated values, based on the regression equation, will be
  written to the Command window. The default is (0), do not create the table.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('polreg')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute polreg
print("Running polreg...")
dm_cmd.polreg(
    in_i='t_assays',  # required
    out_o='t_polreg_out',  # required
    x_f='X',  # required
    y_f='Y',  # required
    order_p='required_val',  # required
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("polreg execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_polreg_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")